# Stage 1 — search

Re-create this stage's script with Gemini's help. The cells below give you the spec, the seed, the gotchas, and a verification step. The implementation itself is yours to write.


## 1. Setup

Every cell in this section is idempotent and safe to re-run. If you opened this notebook fresh (without running Stage 0 first in the same runtime), run all of them now.


### 1a. Clone the repo and `cd` into it


In [ ]:
# Bootstrap: clone the workshop repo into /content and cd into it.
# Idempotent — safe to re-run.
import os, subprocess, sys
REPO_DIR = "/content/ar-bic-2026-workshop"
if not os.path.exists(REPO_DIR):
    subprocess.run(
        ["git", "clone", "--depth", "1",
         "https://github.com/jayprimer/ar-bic-2026-workshop.git", REPO_DIR],
        check=True,
    )
os.chdir(REPO_DIR)
print("cwd:", os.getcwd())


### 1b. Install dependencies

Python (`openai`) and the Node CLI `@llamaindex/liteparse`. First run takes ~30s; re-runs are near-instant.


In [ ]:
# Install dependencies. Idempotent (pip skips already-installed; npm re-link is cheap).
# liteparse only matters for Stage 4 but installing it everywhere keeps each
# notebook self-contained, which is the whole point of re-running this cell.
!pip install -q -r requirements.txt
!npm install -g @llamaindex/liteparse 2>&1 | tail -3


### 1c. (no API key needed for this stage)


In [ ]:
# This stage doesn't call the OpenAI API.


### 1d. Stage the bundle-shipped configs


In [ ]:
# Copy bundle-shipped configs into the directories each stage script expects.
# Each stage's input.txt / criteria.txt / schema.json lives under configs/
# in the repo; the actual scripts read them relative to cwd.
import os, shutil
os.makedirs("stage_01", exist_ok=True)
os.makedirs("stage_02", exist_ok=True)
shutil.copy("configs/stage_01_input.txt",    "stage_01/input.txt")
shutil.copy("configs/stage_02_input.txt",    "stage_02/input.txt")
shutil.copy("configs/stage_02_criteria.txt", "stage_02/criteria.txt")
shutil.copy("configs/schema.json",           "schema.json")
print("configs staged")


## 2. Spec — paste this into Gemini

Open the Gemini side panel in Colab (sparkles icon, top right) and paste the block below as your prompt. Then iterate.

```
Write a Python script (a single file run top-to-bottom) that:

1. Reads PubMed search config from `stage_01/input.txt`. The format is
   `KEY = value` per line, '#' comments, indented continuation lines
   joined with one space. Keys: QUERY (required), N (default 30), TOOL,
   EMAIL.
2. Calls NCBI E-utilities `esearch.fcgi` (db=pubmed, retmax=N,
   retmode=json, sort=date) to get N PMIDs for the query.
3. Calls NCBI E-utilities `efetch.fcgi` (db=pubmed, retmode=xml) for
   those PMIDs to get per-record metadata: pmid, title, abstract,
   authors (list + first_author), journal, year, pub_types.
4. Writes `stage_01/data/pmids.json` with shape:
   `{"query": ..., "n_requested": N, "pmids": [...], "records": [...]}`.
5. Asserts: every record has a non-empty pmid AND a non-empty title.

Use only the standard library (urllib, json, xml.etree.ElementTree).
Do not use Biopython or `requests`.
```


## 3. Gotchas Gemini probably won't know

Copy any that apply into Gemini if it goes off-track:

- **Abstracts are nested XML.** Use `el.itertext()` joined together,
  NOT `el.text`, when reading `<ArticleTitle>` and `<AbstractText>`.
  `.text` silently truncates at the first inline child (`<i>`,
  `<sub>`, `<sup>`).
- **Abstract can be multi-part labeled.** Find all
  `.//Abstract/AbstractText`, read each `Label` attribute, and join
  them as `"BACKGROUND: ... METHODS: ..."`.
- **efetch can reorder.** Re-sort `records` to match the esearch
  `pmids` order before writing.
- **Identify yourself to NCBI.** Pass `tool` and `email` URL params
  (anonymous clients are throttled hard).


## 4. Seed — a few lines to anchor Gemini in the right direction


In [ ]:
import json, os, urllib.parse, urllib.request
import xml.etree.ElementTree as ET

STAGE = "stage_01"
DATA = f"{STAGE}/data"
os.makedirs(DATA, exist_ok=True)
INPUT_PATH = f"{STAGE}/input.txt"
HEADERS = {"User-Agent": "ar-bic-2026/0.1"}


## 5. Your implementation

Drive Gemini to fill this in. Iterate until the verification cell below passes.


In [ ]:
# TODO: implement Stage 1 here.
# Read the spec above. Use the seed cell's imports.
# When done, run the verification cell next.


## 6. Verify


In [ ]:
import json, os
assert os.path.exists("stage_01/data/pmids.json"), "no output file"
d = json.load(open("stage_01/data/pmids.json"))
assert "pmids" in d and "records" in d, "missing keys"
assert len(d["pmids"]) == len(d["records"]), "pmid/record count mismatch"
for r in d["records"]:
    assert r.get("pmid"), "record missing pmid"
    assert r.get("title"), f"{r.get('pmid')}: missing title"
print(f"OK — {len(d['records'])} records")


## 7. Run the eval grader

The eval reads only your stage's output and writes `stage_01/eval/eval_*.json` + `score.json`.


In [ ]:
!python eval/eval_01_script.py


## 8. Stuck? Skip this stage

Copy the reference run's Stage 1 output into place so the next stage's notebook can still run. Use this sparingly — the point of the workshop is to *re-create* each stage.


In [ ]:
import os, shutil
os.makedirs("stage_01/data", exist_ok=True)
shutil.copy("reference_outputs/stage_01/data/pmids.json",
            "stage_01/data/pmids.json")
print("copied reference Stage 1 output")
